# Fundamentals - OpenAI Agents-style identity

Objetivo: demostrar que `framework="openai-agents"` declara identidad, mientras el Provider resuelto por `toolkit.runtime` conserva la responsabilidad de ejecutar. No se importa un SDK de framework.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| RUN_OPENAI_STYLE_LIVE | 1 | Usa 0 para desactivar la ejecucion async live. |
| provider | auto | Resolver provider mediante toolkit.runtime. |
| framework | openai-agents | Aplicar el estilo de integracion sin SDK directo. |

La seleccion de framework es style-only; el contrato de runtime y RunResult permanece estable.

## 1) Frontera observable

```text
Provider  = backend que ejecuta
Framework = identidad declarativa
RunResult = contrato comun observable
```

La ejecucion live requiere un flag explicito. Un skip nunca se presenta como ejecucion.

In [ ]:
import os

import agentic_systems as toolkit

RUN_LIVE = os.getenv("RUN_OPENAI_STYLE_LIVE", "1").strip().lower() in {"1", "true", "yes"}
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=1, max_turns=3)
runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
runtime_description = runtime.describe()
resolved_provider = runtime_description.get("selected_provider")
framework_profile = toolkit.integrations.framework_profile("openai-agents")

toolkit.show_json({
    "run_live": RUN_LIVE,
    "runtime": runtime_description,
    "framework": framework_profile.to_dict(),
}, title="Provider y framework")

## 2) Tool y agente mediante la API publica

La tool consulta la superficie instalada; no contiene una respuesta precocinada. El mismo agente puede ejecutarse con cualquier Provider compatible.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.PUBLIC_API,
        "package_version": toolkit.__version__,
    }

REQUEST = "Usa inspect_public_api para verificar si graph es parte de la API publica."
READY = RUN_LIVE and bool(resolved_provider and resolved_provider != "auto")

if READY:
    system = toolkit.system(runtime=runtime)
    agent = system.agent(
        name="openai_agents_identity_probe",
        instructions="Usa la tool requerida y responde solo con evidencia observada.",
        tools=[inspect_public_api],
        engine=resolved_provider,
        runtime=runtime,
        framework="openai-agents",
        contract=toolkit.AgentContract(
            must_call=["inspect_public_api"],
            completion="when_required_tools_satisfied",
        ),
        policy=toolkit.RunPolicy(max_turns=3, max_tool_calls=1, temperature=0.0),
    )
else:
    system = agent = None
    toolkit.show_json({
        "status": "skipped",
        "reason": "Configura un Provider resoluble, o usa RUN_OPENAI_STYLE_LIVE=0.",
        "resolved_provider": resolved_provider,
    }, title="Integration gate")

## 3) Ejecutar y conservar `RunResult`

La identidad de framework queda en metadata. Runtime, tools, validacion y usage siguen perteneciendo al contrato de Agentic Systems.

In [ ]:
if READY:
    result = await agent.arun(REQUEST, mode="eval")
    toolkit.human_result(result, title="OpenAI Agents-style identity RunResult", show_lineage=True)
else:
    result = None

## 4) Extender campos sin crear otro envelope

`fields_mapper` recibe evidencia ya normalizada. El mapper solo anade campos de dominio; no reconstruye runtime, tools, usage ni validacion.

In [ ]:
def evidence_fields(_result, context: dict) -> dict:
    tools = context.get("tools") or []
    return {
        "observed_tool_names": [item.get("name") for item in tools],
        "observed_tool_count": len(tools),
        "evidence_source": "RunResult.tool_events",
    }

if result is not None:
    output = toolkit.agent_output(
        result,
        kind="integration",
        fields_mapper=evidence_fields,
        include_trace=False,
    )
    toolkit.show_json(output, title="Canonical agent_output")
else:
    output = None
    toolkit.show_json({"status": "skipped", "reason": "No existe RunResult live."}, title="agent_output gate")

## 5) Lineage desde evidencia real

Lineage se deriva exclusivamente del `RunResult`; no representa una traza de un SDK externo.

In [ ]:
if result is not None:
    lineage = result.lineage(
        name="tutorial.integration.identity",
        question=REQUEST,
        goal="Explicar provider, framework y evidencia sin mezclarlos.",
    )
    toolkit.show(lineage, title="Lineage Memory")
else:
    lineage = None

## 6) API realmente ejercitada

La lista corresponde a llamadas visibles en las celdas anteriores.

In [ ]:
api_coverage = [
    "toolkit.scheduler",
    "toolkit.runtime",
    "toolkit.integrations.framework_profile",
    "toolkit.tool",
    "toolkit.system",
    "system.agent",
    "agent.arun",
    "toolkit.agent_output(fields_mapper=...)",
    "RunResult.lineage",
    "toolkit.human_result",
    "toolkit.show_json",
]

toolkit.show_json(api_coverage, title="Integration API coverage")

## Lectura correcta

`framework="openai-agents"` es identidad declarativa en 1.1. La ejecucion pertenece al Provider seleccionado y la salida pertenece a `RunResult`.